* [Snabbcheck](https://github.com/salgo60/Svenskabadplatser/blob/main/Jupyter/Snabbcheck.ipynb)

In [1]:
from datetime import datetime
start_time  = datetime.now()
print("Last run: ", start_time)

Last run:  2026-07-15 17:05:06.409994


In [2]:
import requests
from collections import Counter

url = "https://badplatsen.havochvatten.se/badplatsen/api/detail"

r = requests.get(url)
r.raise_for_status()

data = r.json()

print(type(data)) 
print("Antal poster:", len(data))

counter = Counter(item.get("euType") for item in data)

print(counter)

<class 'list'>
Antal poster: 2632
Counter({False: 2145, True: 487})


In [3]:
for value, count in sorted(counter.items()):
    print(f"{value}: {count}")

False: 2145
True: 487


In [4]:
import requests
from datetime import date

# -------------------------------------------------
# Hämta HaV badplatser
# -------------------------------------------------

HAV_URL = "https://badplatsen.havochvatten.se/badplatsen/api/detail"

print("Loading HaV data...")
hav = requests.get(HAV_URL).json()

print(f"HaV badplatser: {len(hav)}")

# indexera på nutsCode
hav_index = {}
for b in hav:
    nuts = b.get("nutsCode")
    if nuts:
        hav_index[nuts] = b

# -------------------------------------------------
# Hämta Wikidata
# -------------------------------------------------

query = """
SELECT ?item ?itemLabel ?EUbath WHERE {
  ?item wdt:P6104 wd:Q106774536 ;
        wdt:P17 wd:Q34 ;
        wdt:P9616 ?EUbath .

  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "sv,en".
  }
}
ORDER BY ?itemLabel
"""

print("Loading Wikidata...")

r = requests.get(
    "https://query.wikidata.org/sparql",
    params={"query": query},
    headers={
        "Accept": "application/sparql-results+json",
        "User-Agent": "EU Bath Checker/1.0"
    },
)

wd = r.json()["results"]["bindings"]

print(f"Wikidata badplatser: {len(wd)}")

# -------------------------------------------------
# HTML
# -------------------------------------------------

today = date.today().strftime("%Y%m%d")
filename = f"EUBathSv{today}.html"

ok = 0
wrong = 0
missing = 0

rows = []

for row in wd:

    qid = row["item"]["value"].split("/")[-1]
    label = row["itemLabel"]["value"]
    eu = row["EUbath"]["value"]

    wd_link = f"https://www.wikidata.org/wiki/{qid}"
    hav_link = f"https://badplatsen.havochvatten.se/badplatsen/karta/#/bath/{eu}"

    if eu not in hav_index:
        colour = "gray"
        status = "Missing in HaV"
        missing += 1

    else:
        euType = hav_index[eu]["euType"]

        if euType:
            colour = "green"
            status = "OK"
            ok += 1
        else:
            colour = "red"
            status = "euType = false"
            wrong += 1

    rows.append(f"""
<li style="color:{colour}">
<a href="{wd_link}">{label}</a>
&nbsp;(<a href="{hav_link}">{eu}</a>)
&nbsp;- {status}
</li>
""")

# -------------------------------------------------
# skriv HTML
# -------------------------------------------------

html = f"""
<!DOCTYPE html>
<html lang="sv">
<head>
<meta charset="utf-8">
<title>EU Bath Sweden</title>
<style>
body {{
    font-family: Arial;
}}
li {{
    margin:4px;
}}
</style>
</head>

<body>

<h1>EU Bath Sweden</h1>

<p>
Datum: {today}<br>
Wikidata: {len(wd)}<br>
HaV: {len(hav)}<br><br>

<b style="color:green">OK:</b> {ok}<br>
<b style="color:red">euType=false:</b> {wrong}<br>
<b style="color:gray">Saknas i HaV:</b> {missing}
</p>

<hr>

<ul>

{''.join(rows)}

</ul>

</body>
</html>
"""

with open(filename, "w", encoding="utf-8") as f:
    f.write(html)

print()
print("Finished.")
print(filename)

Loading HaV data...
HaV badplatser: 2632
Loading Wikidata...
Wikidata badplatser: 806

Finished.
EUBathSv20260715.html
